In [13]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Import FOMO functions

In [14]:
import sys
sys.path.append("../../")
from fomo.patterns.missing_data_matrix import missing_data_matrix, flag_missing_data
from fomo.patterns.missing_data_probabilities import missing_data_probabilities

# Test 1: Known 50% Always

## Make dataset

Artificial dataset: every other value of Missing_Flag is True/False, intervals are 15 minutes

In [15]:
# Creating the datetime range
start_date = "2024-01-01"
end_date = "2024-01-29"
datetime_range = pd.date_range(start=start_date, end=end_date, freq='15T', inclusive='left')

# Creating the dataframe
df = pd.DataFrame({
    "person_id": np.ones(len(datetime_range), dtype=int),  # All values are 1
    "datetime": datetime_range,
    "heart_rate": np.random.uniform(low=60, high=100, size=len(datetime_range)),
    "Missing_Flag": np.tile([False, True], len(datetime_range)//2)
})

# Setting heart_rate to NaN where Missing_Flag is True
df.loc[df['Missing_Flag'], 'heart_rate'] = np.nan

flagged_df_50percent = df
flagged_df_50percent.head()


,person_id,datetime,heart_rate,Missing_Flag
0,1,2024-01-01 00:00:00,70.964272,False
1,1,2024-01-01 00:15:00,NaN,True
2,1,2024-01-01 00:30:00,71.464897,False
3,1,2024-01-01 00:45:00,NaN,True
4,1,2024-01-01 01:00:00,66.972966,False


## Test missing_data_probabilities

Test 1: 30 minute intervals \
Expected output: All values should be 0.5, as long as missingness_interval is an integer multiple of 30

In [16]:
missingness_avg, missingness_var = missing_data_probabilities(flagged_df_50percent, time_column='datetime', axes="day", missingness_interval=30)
missingness_avg

array([0.48214286, 0.5       , 0.5       , 0.5       , 0.5       ,
       0.5       , 0.5       , 0.5       , 0.5       , 0.5       ,
       0.5       , 0.5       , 0.5       , 0.5       , 0.5       ,
       0.5       , 0.5       , 0.5       , 0.5       , 0.5       ,
       0.5       , 0.5       , 0.5       , 0.5       , 0.5       ,
       0.5       , 0.5       , 0.5       , 0.5       , 0.5       ,
       0.5       , 0.5       , 0.5       , 0.5       , 0.5       ,
       0.5       , 0.5       , 0.5       , 0.5       , 0.5       ,
       0.5       , 0.5       , 0.5       , 0.5       , 0.5       ,
       0.5       , 0.5       , 0.5       ])

# Test 2: 6am-6pm only

## Make artificial dataset
In this artificial dataset, starting exactly at 6:00am, all values are not missing. Then starting at 18:00, all values are missing. 

In [17]:
# Recreating the datetime range
datetime_range = pd.date_range(start=start_date, end=end_date, freq='15T', inclusive='left')

# Function to set Missing_Flag based on the time condition
def set_missing_flag(hour):
    if hour >= 18  or hour < 6:  # From 6pm to 6am, make data missing (return True)
        return True
    else:
        return False

# Creating the new dataframe from scratch with the adjusted Missing_Flag condition
df_new = pd.DataFrame({
    "person_id": np.ones(len(datetime_range), dtype=int),
    "datetime": datetime_range
})

df_new['Missing_Flag'] = df_new['datetime'].dt.hour.apply(set_missing_flag)
df_new['heart_rate'] = np.random.uniform(low=60, high=100, size=len(datetime_range))
df_new.loc[df_new['Missing_Flag'], 'heart_rate'] = np.nan

flagged_df_midday = df_new
flagged_df_midday.head()


,person_id,datetime,Missing_Flag,heart_rate
0,1,2024-01-01 00:00:00,True,NaN
1,1,2024-01-01 00:15:00,True,NaN
2,1,2024-01-01 00:30:00,True,NaN
3,1,2024-01-01 00:45:00,True,NaN
4,1,2024-01-01 01:00:00,True,NaN


## Test missing_data_probabilities

Test 1: hours \
Expected output: 6 1's, followed by 12 0's, followed by 6 1's

In [18]:
missingness_avg, missingness_var = missing_data_probabilities(flagged_df_midday, time_column='datetime', axes="day", missingness_interval=60)
missingness_avg

array([1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 0.75, 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.25, 1.  , 1.  , 1.  ,
       1.  , 1.  ])

Test 2: 6 hour intervals \
Expected output: [1,0,0,1]

In [19]:
missingness_avg, missingness_var = missing_data_probabilities(flagged_df_midday, time_column='datetime', axes="day", missingness_interval=60*6)
missingness_avg

array([1.        , 0.95833333, 0.        , 0.04166667])

Test 3: Week Level, 24 hour intervals \
Expected output: all values should be 0.5

In [20]:
missingness_avg, missingness_var = missing_data_probabilities(flagged_df_midday, time_column='datetime', axes="week", missingness_interval=60*24)
missingness_avg

array([0.625, 0.5  , 0.5  , 0.5  , 0.5  , 0.5  , 0.5  ])

Test 4 day x week, 6 hours \
Expected output: [1, 0, 0, 1] repeated 7 times

In [21]:
missingness_avg, missingness_var = missing_data_probabilities(flagged_df_midday, time_column='datetime', axes=['day', "week"], missingness_interval=60*6)
missingness_avg

array([[1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 0.95833333, 0.        , 0.04166667]])

## Test 3: Saturday/Sunday Dependencies

## Make dataset
This dataset is similar to test 2 (always missing between 6pm and 6am), except 
* Values are ALL missing on Saturdays
* On Sundays, values are only present noon-6pm

In [22]:
datetime_range = pd.date_range(start=start_date, end=end_date, freq='15T', inclusive='left')

# Function to set Missing_Flag based on the new conditions (Saturday, Sunday with specific hours)
def set_advanced_missing_flag(row):
    if row.dayofweek == 5:  # Saturday
        return True
    elif row.dayofweek == 6:  # Sunday
        # True except for noon to 6pm
        if 12 <= row.hour < 18:
            return False
        else:
            return True
    else:
        if row.hour >= 18 or row.hour < 6:  # From 6pm to 6am
            return True
        else:
            return False

# Creating the new dataframe from scratch with the updated Missing_Flag condition
df_advanced = pd.DataFrame({
    "person_id": np.ones(len(datetime_range), dtype=int),
    "datetime": datetime_range
})

df_advanced['Missing_Flag'] = df_advanced['datetime'].apply(set_advanced_missing_flag)
df_advanced['heart_rate'] = np.random.uniform(low=60, high=100, size=len(datetime_range))
df_advanced.loc[df_advanced['Missing_Flag'], 'heart_rate'] = np.nan

flagged_df_weekend = df_advanced
flagged_df_weekend.head()

weekend_df_val = flagged_df_weekend.copy()
weekend_df_val['date'] = weekend_df_val['datetime'].dt.date

print('1/6 is a Saturday, so that should be equal to 1.0, 1/7 should be 0.75')
print((weekend_df_val.groupby('date')['Missing_Flag'].sum() / 96).head(7))


1/6 is a Saturday, so that should be equal to 1.0, 1/7 should be 0.75
date
2024-01-01    0.50
2024-01-02    0.50
2024-01-03    0.50
2024-01-04    0.50
2024-01-05    0.50
2024-01-06    1.00
2024-01-07    0.75
Name: Missing_Flag, dtype: float64


## Test missing_data_probabilities

Test 1: Simple week (24 hour intervals) \
Expected output: All values should be 0.5, but the day corresponding to saturday should be 1.0, and the day corresponding to Sunday should be 0.75 \
Without deliberate sorting by weekdays, I think the correct output is [0.5, 0.5, 0.5, 0.5, 0.5, 1.0, 0.75] \
If you're sorting by weekdays, the correct output is [0.75, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0]

In [23]:
missingness_avg, missingness_var = missing_data_probabilities(flagged_df_weekend, time_column='datetime', axes="week", missingness_interval=60*24)
missingness_avg

array([0.8125, 0.5   , 0.5   , 0.5   , 0.5   , 0.5   , 1.    ])

Test 2: day x week (6 hour intervals) \
Expected output: All weekdays should be [1, 0, 0, 1], Sunday should be [1, 1, 0, 1], and Saturday should be [1, 1, 1, 1]

In [24]:
missingness_avg, missingness_var = missing_data_probabilities(flagged_df_weekend, time_column='datetime', axes=['day', "week"], missingness_interval=60*6)
missingness_avg

array([[1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 0.95833333, 0.        , 0.04166667],
       [1.        , 1.        , 1.        , 1.        ],
       [1.        , 1.        , 0.95833333, 0.04166667]])